# ImageAsset Deep Dive – Static Images in Programmatic Video Editing

---

Images are everywhere in video production. A logo in the corner. A title card at the start. A photo montage sequence. A watermark that stays visible throughout. Static images add branding, context, and visual interest to video compositions.

In this notebook, we'll explore every aspect of ImageAsset in the VideoDB Editor—from basic display to sophisticated layering, effects, and animations. You'll learn how to control sizing, positioning, cropping, opacity, filters, transitions, and timing to create professional image overlays.

**What you'll learn:**

- How ImageAsset works and differs from VideoAsset (no temporal properties like `start` or `volume`)
- The complete property set: `id` (required) and `crop` (optional)
- Clip-level control: `duration`, `fit`, `position`, `offset`, `scale`, `opacity`, `filter`, `transition`
- All 4 fit modes and when to use each (crop, contain, cover, none)
- The 9-position grid system plus fine-tuning with offset
- Cropping edges to focus on specific image regions
- Visual effects: opacity, filters, and fade transitions
- Image sequences and slideshows with smooth transitions
- Layering images over video for complex compositions
- Real-world patterns: watermarks, title cards, Ken Burns effects

By the end, you'll have complete mastery of ImageAsset for adding static graphics, watermarks, title cards, and creative overlays to your programmatic video projects.

---
## 📦 Step 1: Install VideoDB Editor SDK

Run the next cell to install the VideoDB Editor SDK. This will take a moment.

In [ ]:
!pip -q install videodb


---
## 📦 Step 2: Import Required Modules

We'll import the core VideoDB modules and all the Editor objects we need for working with images.

In [ ]:
import videodb
import os
from getpass import getpass
from videodb import play_stream
from videodb.editor import (
    Timeline, Track, Clip,
    VideoAsset, ImageAsset,
    Fit, Position, Offset, Transition, Crop, Filter
)

---
## 📦 Step 3: Connect to VideoDB

Run the next cell and enter your VideoDB API key when prompted. This establishes a connection to your VideoDB collection.

In [ ]:
api_key = getpass("Please enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
coll = conn.get_collection()

print("✅ Connected to VideoDB")

---
## 📦 Step 4: Upload Assets

We'll upload one background video (for layering demonstrations) and three images with different aspect ratios:
- **Landscape image** (wide, e.g., 16:9)
- **Portrait image** (tall, e.g., 9:16)
- **Square image** (e.g., 1:1)

These different aspect ratios will help us understand how fit modes work.

**Note:** Replace the URLs with your own image URLs. Make sure they match the aspect ratios mentioned in the comments.

In [ ]:
# Upload background video (calm ambience, won't be the focus)
video = coll.upload(url="https://www.youtube.com/watch?v=wU0PYcCsL6o")
print(f"✅ Uploaded video: {video.id}")

In [ ]:
# Upload landscape image (16:9 or similar wide aspect ratio)
# Replace with your landscape image URL
landscape_image = coll.upload(url="https://images.unsplash.com/photo-1506905925346-21bda4d32df4")
print(f"✅ Uploaded landscape image: {landscape_image.id}")

# If you've already uploaded this image, uncomment and use:
# landscape_image = coll.get_image("your_image_id_here")

In [ ]:
# Upload portrait image (9:16 or similar tall aspect ratio)
# Replace with your portrait image URL
portrait_image = coll.upload(url="https://images.unsplash.com/photo-1682687220742-aba13b6e50ba")
print(f"✅ Uploaded portrait image: {portrait_image.id}")

# If you've already uploaded this image, uncomment and use:
# portrait_image = coll.get_image("your_image_id_here")

In [ ]:
# Upload square image (1:1 aspect ratio)
# Replace with your square image URL
square_image = coll.upload(url="https://images.unsplash.com/photo-1579546929518-9e396f3cc809")
print(f"✅ Uploaded square image: {square_image.id}")

# If you've already uploaded this image, uncomment and use:
# square_image = coll.get_image("your_image_id_here")

In [ ]:
# Upload logo image (for watermark demonstrations)
# Replace with your logo/icon image URL (preferably with transparency or small aspect ratio)
logo_image = coll.upload(url="https://images.unsplash.com/photo-1611162617474-5b21e879e113")
print(f"✅ Uploaded logo image: {logo_image.id}")

# If you've already uploaded this image, uncomment and use:
# logo_image = coll.get_image("your_image_id_here")

---
## 📦 Step 5: Basic ImageAsset Display

Let's start simple. We'll display a single image for 5 seconds.

Notice that:
- `ImageAsset` only requires an `id` (no `start` or `volume` like VideoAsset)
- `duration` is specified in the `Clip`, not the asset
- We use a neutral gray background (`#2B2B2B`) as our timeline canvas

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=5
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What just happened?**

We created a timeline with a single image that displays for 5 seconds. By default, the image uses `Fit.crop`, which scales it to fill the viewport while maintaining aspect ratio (cropping edges if needed).

---
## 📦 Step 6: Understanding Fit Modes

The `fit` parameter controls how an image scales to match the timeline resolution. There are 4 modes:

- **`Fit.crop`** (default): Fills the viewport, crops edges if aspect ratios differ. Best for backgrounds.
- **`Fit.contain`**: Shows entire image with letterboxing (black bars). Preserves aspect ratio.
- **`Fit.cover`**: Stretches to fill viewport (may distort). Ignores aspect ratio.
- **`Fit.none`**: Preserves original pixel dimensions.

Let's demonstrate all 4 modes with the same landscape image, displaying each for 3 seconds.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Fit.crop (default) - fills viewport, crops edges
clip_crop = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    fit=Fit.crop
)

# Fit.contain - shows entire image, adds letterboxing
clip_contain = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    fit=Fit.contain
)

# Fit.cover - stretches to fill (may distort)
clip_cover = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    fit=Fit.cover
)

# Fit.none - preserves original dimensions
clip_none = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    fit=Fit.none
)

track = Track()
track.add_clip(0, clip_crop)      # 0-3s
track.add_clip(3, clip_contain)   # 3-6s
track.add_clip(6, clip_cover)     # 6-9s
track.add_clip(9, clip_none)      # 9-12s

timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we're seeing:**

- **0-3s (crop)**: Image fills the screen, edges may be cropped
- **3-6s (contain)**: Entire image visible, black bars on sides
- **6-9s (cover)**: Image stretched to fill (notice distortion)
- **9-12s (none)**: Original pixel size (may be larger or smaller than viewport)

**Tip:** Use `Fit.crop` for backgrounds, `Fit.contain` for showcasing full images, and `Fit.none` for logos/overlays.

---
## 📦 Step 7: Position Presets

The `position` parameter places images in one of 9 preset zones. This is essential for layouts like corner logos, bottom captions, or centered titles.

The 9 positions are:
- `top_left`, `top`, `top_right`
- `left`, `center` (default), `right`
- `bottom_left`, `bottom`, `bottom_right`

Let's display our square image in each position for 3 seconds using `Fit.none` so we can clearly see the positioning.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

positions = [
    Position.top_left,
    Position.top,
    Position.top_right,
    Position.right,
    Position.bottom_right,
    Position.bottom,
    Position.bottom_left,
    Position.left,
    Position.center
]

track = Track()

for i, pos in enumerate(positions):
    clip = Clip(
        asset=ImageAsset(id=square_image.id),
        duration=3,
        fit=Fit.none,
        scale=0.3,  # Make it smaller to see positioning clearly
        position=pos
    )
    track.add_clip(i * 3, clip)  # Each position for 3 seconds

timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we're seeing:**

The image moves through all 9 positions, spending 3 seconds in each location. This demonstrates the grid system that Position provides for quick layout control.

---
## 📦 Step 8: Fine-Tuning with Offset

While `position` gives us 9 presets, `offset` allows pixel-perfect adjustments. Offset values are relative to viewport dimensions:
- `x`: -1.0 to 1.0 (negative = left, positive = right)
- `y`: -1.0 to 1.0 (negative = up, positive = down)

Example: `Offset(x=0.1)` moves the image 10% of viewport width to the right.

Let's create a typical watermark pattern: top-right position with slight offset for padding.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Add background video
video_clip = Clip(
    asset=VideoAsset(id=video.id),
    duration=10
)

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

# Add watermark with offset
watermark_clip = Clip(
    asset=ImageAsset(id=logo_image.id),
    duration=10,
    position=Position.top_right,
    fit=Fit.none,
    scale=0.15,
    offset=Offset(x=-0.05, y=0.05),  # Pull back from edge
    opacity=0.8  # Slightly transparent
)

watermark_track = Track()
watermark_track.add_clip(0, watermark_clip)
timeline.add_track(watermark_track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we built:**

A professional watermark overlay! The logo sits in the top-right corner with:
- `offset` pulling it slightly away from the edges (5% padding)
- `scale=0.15` making it 15% of original size
- `opacity=0.8` making it semi-transparent

This is a common pattern in branded video content.

---
## 📦 Step 9: Scaling Images

The `scale` parameter resizes images as a multiplier:
- `scale=1.0` (default): Original size
- `scale=0.5`: Half size
- `scale=2.0`: Double size

Range: 0.0 to 10.0

Let's demonstrate different scale values with the same image.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

scales = [0.2, 0.5, 1.0, 1.5]

track = Track()

for i, scale_value in enumerate(scales):
    clip = Clip(
        asset=ImageAsset(id=portrait_image.id),
        duration=3,
        fit=Fit.none,
        scale=scale_value
    )
    track.add_clip(i * 3, clip)

timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we're seeing:**

The image scales from tiny (0.2x) to large (1.5x). Notice how `Fit.none` preserves the aspect ratio while scale changes the overall size.

**Tip:** Combine `scale` with `position` for corner logos, or use larger scales with `position` and `offset` to create zoom/pan effects.

---
## 📦 Step 10: Cropping Image Edges

The `crop` parameter trims edges of the source image before placing it in the clip. Crop values are relative (0.0 to 1.0):
- `Crop(left=0.25)`: Removes 25% from the left edge
- `Crop(top=0.1)`: Removes 10% from the top

This is useful for focusing on specific regions of an image or removing unwanted edges.

Let's demonstrate cropping all four edges sequentially.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Original (no crop)
clip_original = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3
)

# Crop left edge
clip_crop_left = Clip(
    asset=ImageAsset(
        id=landscape_image.id,
        crop=Crop(left=0.3)
    ),
    duration=3
)

# Crop right edge
clip_crop_right = Clip(
    asset=ImageAsset(
        id=landscape_image.id,
        crop=Crop(right=0.3)
    ),
    duration=3
)

# Crop top edge
clip_crop_top = Clip(
    asset=ImageAsset(
        id=landscape_image.id,
        crop=Crop(top=0.3)
    ),
    duration=3
)

# Crop bottom edge
clip_crop_bottom = Clip(
    asset=ImageAsset(
        id=landscape_image.id,
        crop=Crop(bottom=0.3)
    ),
    duration=3
)

track = Track()
track.add_clip(0, clip_original)       # 0-3s: Original
track.add_clip(3, clip_crop_left)      # 3-6s: Left cropped
track.add_clip(6, clip_crop_right)     # 6-9s: Right cropped
track.add_clip(9, clip_crop_top)       # 9-12s: Top cropped
track.add_clip(12, clip_crop_bottom)   # 12-15s: Bottom cropped

timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we're seeing:**

Each segment shows a different crop:
- **0-3s**: Original image
- **3-6s**: 30% cropped from left (right side remains)
- **6-9s**: 30% cropped from right (left side remains)
- **9-12s**: 30% cropped from top (bottom remains)
- **12-15s**: 30% cropped from bottom (top remains)

**Use case:** Focusing on a specific region of an image, like a portrait within a group photo.

---
## 📦 Step 11: Opacity and Transparency

The `opacity` parameter controls transparency:
- `1.0` (default): Fully opaque
- `0.5`: 50% transparent
- `0.0`: Fully invisible

This is useful for subtle overlays, watermarks, or blending multiple image layers.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Base layer: video
video_clip = Clip(
    asset=VideoAsset(id=video.id),
    duration=12
)

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

# Overlay track with varying opacity
overlay_track = Track()

opacities = [1.0, 0.7, 0.4, 0.1]

for i, opacity_value in enumerate(opacities):
    clip = Clip(
        asset=ImageAsset(id=square_image.id),
        duration=3,
        opacity=opacity_value
    )
    overlay_track.add_clip(i * 3, clip)

timeline.add_track(overlay_track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we're seeing:**

The image overlay becomes progressively more transparent over the video:
- **0-3s**: Fully opaque (1.0) - image completely covers video
- **3-6s**: 70% opaque - video shows through
- **6-9s**: 40% opaque - mostly transparent
- **9-12s**: 10% opaque - barely visible

**Tip:** Use opacity=0.6-0.8 for subtle watermarks that don't overwhelm the content.

---
## 📦 Step 12: Applying Filters

Filters apply color treatments to images. Available filters:
- `greyscale`: Remove color
- `blur`: Blur the image
- `contrast`: Increase contrast
- `darken`: Darken the scene
- `lighten`: Lighten the scene
- `boost`: Boost contrast and saturation
- `muted`: Reduce saturation and contrast
- `negative`: Invert colors

Let's demonstrate a few filters on the same image.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Original (no filter)
clip_original = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3
)

# Greyscale
clip_greyscale = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    filter=Filter.greyscale
)

# Boost (enhanced colors)
clip_boost = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    filter=Filter.boost
)

# Negative (inverted colors)
clip_negative = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    filter=Filter.negative
)

track = Track()
track.add_clip(0, clip_original)   # 0-3s: Original
track.add_clip(3, clip_greyscale)  # 3-6s: Greyscale
track.add_clip(6, clip_boost)      # 6-9s: Boost
track.add_clip(9, clip_negative)   # 9-12s: Negative

timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we're seeing:**

The same image rendered with different color treatments. Filters work identically on both VideoAsset and ImageAsset.

**Tip:** Use `Filter.greyscale` for vintage effects, `Filter.boost` for vibrant social media content, or `Filter.muted` for subtle backgrounds.

---
## 📦 Step 13: Transitions (Fade In/Out)

The `transition` parameter adds smooth entry and exit animations. For images, fade transitions create professional-looking slideshows.

Transition properties:
- `in_`: Entry animation ("fade")
- `out`: Exit animation ("fade")
- `duration`: How long the transition lasts (in seconds)

**Note:** Use `in_` (with underscore) because `in` is a Python keyword.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=ImageAsset(id=portrait_image.id),
    duration=8,
    transition=Transition(in_="fade", out="fade", duration=2)
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we're seeing:**

The image fades in over 2 seconds, displays for 4 seconds at full opacity, then fades out over 2 seconds.

Timeline breakdown:
- **0-2s**: Fade in
- **2-6s**: Full visibility
- **6-8s**: Fade out

This creates smooth, professional transitions between content.

---
## 📦 Step 14: Image Slideshow

Let's combine multiple images with fade transitions to create a slideshow. Each image will fade in, display, and fade out seamlessly.

We'll use all three of our uploaded images.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

images = [landscape_image, portrait_image, square_image]

track = Track()

for i, img in enumerate(images):
    clip = Clip(
        asset=ImageAsset(id=img.id),
        duration=5,
        transition=Transition(in_="fade", out="fade", duration=1)
    )
    track.add_clip(i * 5, clip)

timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we built:**

A 15-second slideshow with three images:
- Each image displays for 5 seconds
- 1-second fade transitions create smooth blending
- Images are placed sequentially on the timeline

This pattern is common in photo galleries, intros, and outros.

---
## 📦 Step 15: Ken Burns Effect (Zoom and Pan)

The Ken Burns effect creates motion in static images by simulating a camera zoom or pan. We achieve this by using multiple clips of the same image with different `scale` and `position` values.

Let's create a zoom-in effect: start small and centered, then zoom to 1.5x while maintaining center position.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Start: Small scale
clip_start = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    scale=1.0,
    fit=Fit.crop,
    position=Position.center,
    transition=Transition(out="fade", duration=0.5)
)

# Middle: Medium scale
clip_middle = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    scale=1.3,
    fit=Fit.crop,
    position=Position.center,
    transition=Transition(in_="fade", out="fade", duration=0.5)
)

# End: Larger scale (zoomed in)
clip_end = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    scale=1.6,
    fit=Fit.crop,
    position=Position.center,
    transition=Transition(in_="fade", duration=0.5)
)

track = Track()
track.add_clip(0, clip_start)    # 0-3s
track.add_clip(2.5, clip_middle) # 2.5-5.5s (overlap for smooth transition)
track.add_clip(5, clip_end)      # 5-8s

timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we built:**

A simulated zoom effect using three clips with progressively larger `scale` values:
- Clip 1: scale=1.0 (normal)
- Clip 2: scale=1.3 (30% larger)
- Clip 3: scale=1.6 (60% larger)

The overlapping timings with fade transitions create the illusion of smooth zooming. This technique brings static images to life!

**Tip:** Combine with `position` changes to create pan effects (e.g., `Position.left` → `Position.right`).

---
## 📦 Step 16: Layering Multiple Images

Tracks stack vertically—later tracks render on top. This allows us to create complex compositions with multiple image layers.

Let's build a composition with:
- Background video (Track 1)
- Full-screen image overlay with opacity (Track 2)
- Corner logo (Track 3)

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Track 1: Base video layer
video_clip = Clip(
    asset=VideoAsset(id=video.id),
    duration=10
)

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

# Track 2: Semi-transparent overlay
overlay_clip = Clip(
    asset=ImageAsset(id=square_image.id),
    duration=10,
    opacity=0.3,
    filter=Filter.boost
)

overlay_track = Track()
overlay_track.add_clip(0, overlay_clip)
timeline.add_track(overlay_track)

# Track 3: Logo watermark
logo_clip = Clip(
    asset=ImageAsset(id=logo_image.id),
    duration=10,
    position=Position.bottom_right,
    fit=Fit.none,
    scale=0.2,
    offset=Offset(x=-0.05, y=-0.05),
    opacity=0.9
)

logo_track = Track()
logo_track.add_clip(0, logo_clip)
timeline.add_track(logo_track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we built:**

A three-layer composition:
1. **Bottom layer**: Background video
2. **Middle layer**: Semi-transparent image overlay (30% opacity) with color boost
3. **Top layer**: Watermark logo in bottom-right corner

This demonstrates the power of track layering for creating complex visual hierarchies.

**Tip:** Tracks added later always render on top. Use this to control z-order in your compositions.

---
## 📦 Step 17: Timing Control with Timeline Positioning

So far, we've placed clips at the start of the timeline (0 seconds) or sequentially. But `track.add_clip(start, clip)` lets us place images at any timeline position.

This is crucial for:
- Timed overlays (e.g., logo appears at 5 seconds)
- Staggered animations
- Synchronizing with audio/video events

Let's create a video with images appearing at different times.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Base video
video_clip = Clip(
    asset=VideoAsset(id=video.id),
    duration=15
)

video_track = Track()
video_track.add_clip(0, video_clip)
timeline.add_track(video_track)

# Image track with timed appearances
image_track = Track()

# Image 1: Appears at 2 seconds, lasts 3 seconds
clip1 = Clip(
    asset=ImageAsset(id=square_image.id),
    duration=3,
    position=Position.top_left,
    fit=Fit.none,
    scale=0.25,
    transition=Transition(in_="fade", out="fade", duration=0.5)
)
image_track.add_clip(2, clip1)  # Starts at 2s

# Image 2: Appears at 6 seconds, lasts 3 seconds
clip2 = Clip(
    asset=ImageAsset(id=portrait_image.id),
    duration=3,
    position=Position.top_right,
    fit=Fit.none,
    scale=0.25,
    transition=Transition(in_="fade", out="fade", duration=0.5)
)
image_track.add_clip(6, clip2)  # Starts at 6s

# Image 3: Appears at 10 seconds, lasts 3 seconds
clip3 = Clip(
    asset=ImageAsset(id=landscape_image.id),
    duration=3,
    position=Position.bottom,
    fit=Fit.contain,
    scale=0.4,
    transition=Transition(in_="fade", out="fade", duration=0.5)
)
image_track.add_clip(10, clip3)  # Starts at 10s

timeline.add_track(image_track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we built:**

A 15-second video with three images appearing at specific times:
- **2-5s**: Square image in top-left
- **6-9s**: Portrait image in top-right
- **10-13s**: Landscape image at bottom

This demonstrates precise timing control for creating dynamic compositions.

**Use case:** Product showcases, timed annotations, or synchronized visual elements.

---
## 📦 Step 18: Real-World Pattern – Title Card

Title cards are full-screen images (often with text baked in) that introduce videos. They typically:
- Fill the entire viewport
- Display for 3-5 seconds
- Use fade transitions

Let's create a title card followed by main content.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

track = Track()

# Title card: 5 seconds with fade in/out
title_clip = Clip(
    asset=ImageAsset(id=square_image.id),
    duration=5,
    fit=Fit.crop,
    transition=Transition(in_="fade", out="fade", duration=1)
)
track.add_clip(0, title_clip)

# Main video content: starts at 5 seconds
video_clip = Clip(
    asset=VideoAsset(id=video.id),
    duration=10
)
track.add_clip(5, video_clip)

timeline.add_track(track)

stream_url = timeline.generate_stream()
play_stream(stream_url)

**What we built:**

A professional video opening:
- **0-5s**: Title card (image) with fade transitions
- **5-15s**: Main video content

This pattern is common in tutorials, presentations, and social media videos.

**Tip:** Combine with TextAsset for dynamic titles, or use ImageAsset with pre-designed graphics.

---
## 🎯 Wrap-Up

Congratulations! We've explored ImageAsset in depth. Here's what we learned:

### Core Concepts
- **ImageAsset** has only two properties: `id` (required) and `crop` (optional)
- All display properties (duration, position, size, effects) are controlled at the **Clip level**
- Images have no temporal source properties (no `start` or `volume` like VideoAsset)

### Display Control
- **Fit modes** determine scaling: `crop` (fill), `contain` (letterbox), `cover` (stretch), `none` (original)
- **Position** provides 9 preset locations, **offset** fine-tunes placement
- **Scale** resizes images (0.1 for small logos, 1.5+ for zoom effects)
- **Crop** trims edges before rendering (left, right, top, bottom)

### Visual Effects
- **Opacity** controls transparency (0.0-1.0)
- **Filters** apply color treatments (greyscale, boost, blur, etc.)
- **Transitions** create smooth fades (in/out)

### Composition Patterns
- **Watermarks**: `position=top_right`, small `scale`, `opacity=0.8`, `offset` for padding
- **Title cards**: Full-screen with `Fit.crop`, fade transitions
- **Slideshows**: Sequential images with fade transitions
- **Ken Burns**: Multiple clips with progressive `scale` and `position` changes
- **Layering**: Multiple tracks for complex visual hierarchies

### Key Takeaways
1. Use `Fit.crop` for backgrounds, `Fit.none` for logos
2. Combine `position` + `offset` for precise placement
3. Track order determines z-index (later tracks render on top)
4. `track.add_clip(start, clip)` controls when images appear on the timeline

### What's Next?

Now that you understand ImageAsset, try:
- Building photo montages with multiple layers
- Creating animated slideshows with different transition timings
- Combining images with TextAsset for dynamic titles
- Experimenting with filter combinations and opacity blending

ImageAsset is a powerful tool for branding, storytelling, and visual composition. Use it to add static graphics, watermarks, title cards, and creative overlays to your programmatic video projects!

Happy editing! 🎬